### Middle Wares

middleware is a mechanism that allows you to control, monitor, modify, and customize how an AI agent behaves while it is executing.

Think of middleware as a security guard or checkpoint between the user and the AI agent.

It can inspect inputs, modify data, control tool execution, handle errors, and inspect outputs.

Middleware is a layer of logic that runs during an AI agent's execution to monitor, modify, or control its behavior.

Middleware is useful :

- Tracking agent behaviour with logging ,analytics and debugging
- Transforming prompts ,tool selection and output formatting
- Adding retries,fallbacks and early termination logic
- Applying ratelimits,guardrills and PII detection

Imagine you're building a customer support AI agent.

A user asks:
"Check my order status."

Before the AI responds, you may want to:
- Check whether the user is authorized.
- Log the request.
- Limit the number of tool calls.
- Handle model errors.
- Review the AI's response before returning it.

Instead of writing all this logic inside your agent, you can use Middleware.

#### Built in Middlewares in LangChain

LangChain also provides prebuilt middleware for common requirements.

Middleware	Purpose

- SummarizationMiddleware	> Summarizes long conversations
- ModelCallLimitMiddleware	> Limits LLM calls
- ToolCallLimitMiddleware	> Limits tool calls
- ModelRetryMiddleware	> Retries failed model calls
- ToolRetryMiddleware	> Retries failed tool calls
- HumanInTheLoopMiddleware	> Requests human approval for selected tool operations
- PIIMiddleware	> Helps detect and handle sensitive information

These reduce the amount of custom code you need to write.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

##### Summarization Middelware

Summarization Middleware automatically summarizes older conversation messages when the conversation becomes too long.

Imagine you're building an AI chatbot, and a user has been chatting with it for 2 hours.

The conversation contains 100 messages.

Instead of repeatedly sending all 100 messages to the LLM, Summarization Middleware can compress older messages into a shorter summary while keeping recent messages available.

2. Why do we need Summarization Middleware?

Without summarization, a long conversation may cause several problems.

Problem	               How summarization helps

Too many tokens	    --------    Compresses older conversation history

Higher token cost	--------    Can reduce tokens sent on subsequent calls

Slow responses	    --------    May reduce processing time for long histories

Context window limits ------	Helps keep conversation context within limits

Long chat history	 --------   Retains a condensed version of older information

Important: Summarization is lossy. The summary may omit details, so it should not be treated as a perfect replacement for the original messages

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
# from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langchain_core.messages import SystemMessage,HumanMessage


agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.6-flash",  #LLM used to generate the conversation summary
            trigger=("messages",10),  # Summarize when the context reaches approximately 1,000 tokens
            keep=("messages",4) # Preserve the most recent four messages where possible
        )
    ]
)

In [10]:
# Run with the threads

config = {"configurable":{"thread_id":"test-1"}}

In [11]:
questions =[
    "What is 2+2?",
    "what is 10*5?",
    "what is 100/5?",
    "what is 1000-500?",
    "What is 1000+500?",
    "what is 1000*500?",
]

In [13]:
for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")


Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f307573a-2a92-453b-92ce-c6ca393111bc'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'Eo0DCooDAWkUfRNsxq5+iOJ8viAe7BsszksOeQ940N1RzsLy+l8okVwGRjw41UxD/pPUYq2XvVdag0YN17k23P/GbrPHaN9XCLRvvNdZshu43+iOGM2p/an/ZGaRJcDjVVTVYwFP59abZKxBEhq2YHUXWdctCf1xxp/Ph+t5KKmmMgV/D4YXFqFosPx4ZdgdXc+6KRQaYFp+5STtnrfC4uUHxl+E/bo/iOLcVe+TsmX5DKhbXdTRu11l3Nha9pxqSwYzi1nTMokYWrwYKtma5y4n8Eg2UuzLOlZB+JyVuV31utjO+0xj8OB01oOZe8MHKRrYOpCvWNN43H55iv40X5KiBYgHS8nHOnEx0bJrL9dXzj8/MjgColVhxWtkoulsURmhSziKIIufcw74/J2HJ9jYdGzvjbGqR4MihnJ3GanTOm45lKvbC3mSR9JoJ4W86yPpUEUwW2WgPlKXTYeYpY4ep9czTIBwBzSOjB/6SLlNJzHKVOaZFWl1ZbxJqThOw5JanNJfJB2EOWs3ZAhaNQ=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1d8af-fa8e-7708-b070-2dfc1b4d09df-0', tool_calls=[], inv

### TOken Sizw

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage,HumanMessage
from langchain.tools import tool


@tool
def search_hotels(city:str):
    """Search hotels - returns a long response to use more tokens"""
    return f"""Hotels in {city}
    1.Grand Hotel - 5 stars,$200 per night, swimming pool, spa, restaurant
    2.City Inn - 3 stars, $100 per night, free breakfast, gym
    3.Budget Stay - 2 stars, $50 per night, free parking, no restaurant"""


agent1 = create_agent(
    model="groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",  #LLM used to generate the conversation summary
            trigger=("tokens",500),  # Summarize when the context reaches approximately 1,000 tokens
            keep=("tokens",200), # Preserve the most recent four messages where possible
        ),
    ]
)


config = {"configurable":{"thread_id":"test-2"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  #4 chars is 1 token

In [27]:
cities = ["Paris","London","New York","Tokyo","Sydney","Berlin"]

for city in cities:
    response = agent1.invoke(
        {"messages": [HumanMessage(content=f"Search hotels in {city}")]},
        config
    )

    tokens = count_tokens(response['messages'])
    print(f"{city}: ~{tokens} tokens,{len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~162 tokens,4 messages
[HumanMessage(content='Search hotels in Paris', additional_kwargs={}, response_metadata={}, id='c34ddd80-01c0-42ed-a363-11d7e2f58deb'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kbb4wft11', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 280, 'total_tokens': 307, 'completion_time': 0.070731376, 'completion_tokens_details': None, 'prompt_time': 0.019550491, 'prompt_tokens_details': None, 'queue_time': 0.056417106, 'total_time': 0.090281867}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1d8dd-6eef-74a5-bc7b-9f9dfcc053bd-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'kbb4wft11', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i

BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'searchё_hotels' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<tool_call>\n<function=searchё_hotels>\n<parameter=city>\nNew York\n</parameter>\n</function>\n</tool_call>'}}

## Human in the loop Middleware

Human-in-the-Loop (HITL) Middleware allows a human to review and approve, reject, or modify an AI agent's action before it executes.

Imagine you're building an AI banking assistant.

A user says:
"Transfer ₹10,000 to Rahul."

The AI agent understands the request and decides to call a money-transfer tool.

But should the AI immediately transfer the money?

No. We want a human to approve the transaction first.

That's where Human-in-the-Loop Middleware comes in.

Three decisions in HITL

LangChain's HumanInTheLoopMiddleware supports these review decisions:

Decision	What happens

Approve	------- Execute the proposed tool call

Edit	------- Change the tool-call arguments before execution

Reject	------- Do not execute the proposed call; send rejection feedback to the agent

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Simulate sending an email to a recipient."""
    print(f"TO: {to}")
    print(f"SUBJECT: {subject}")
    print(f"BODY: {body}")

    return "Email simulation completed."
model = "groq:qwen/qwen3.8-27b"

agent2 = create_agent(
    model=model,
    tools=[send_email],
    system_prompt=(
        "You are an email assistant. "
        "When the user requests an email, "
        "prepare it and call send_email."
    ),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {
                    "allowed_decisions": [
                        "approve",
                        "edit",
                        "reject"
                    ]
                }
            }
        )
    ],
    checkpointer=InMemorySaver()
)

In [32]:
config = {
    "configurable": {
        "thread_id": "email_demo_001"
    }
}

result = agent2.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to rahul@example.com "
                    "with subject 'Project Update' "
                    "and body 'The project is completed.'"
                )
            }
        ]
    },
    config=config
)

print(result.get("__interrupt__"))

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'The project is completed.', 'subject': 'Project Update', 'to': 'rahul@example.com'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'The project is completed.', 'subject': 'Project Update', 'to': 'rahul@example.com'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='eea6fc13378bd2f11486ca76120ae617', response_schema=None)]


In [33]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [34]:
agent3 = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve","edit","reject"]
                },
                "read_email_tool": False
            }
        )
    ],
    checkpointer=InMemorySaver()
)

In [35]:
config1 = {"configurable":{"thread_id":"test-approved"}}

res = agent3.invoke({
    "messages":[HumanMessage(content="Send email to kalyantest@gmail.com with Subject 'Hello' and body 'How are you?' ")]
},
config=config1)

In [36]:
res

{'messages': [HumanMessage(content="Send email to kalyantest@gmail.com with Subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='cdd4a78e-3bb6-4a8f-91f5-b19ea86b2210'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 's40hewjsp', 'function': {'arguments': '{"body":"How are you?","recipient":"kalyantest@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 379, 'total_tokens': 439, 'completion_time': 0.190868615, 'completion_tokens_details': None, 'prompt_time': 0.026136365, 'prompt_tokens_details': None, 'queue_time': 0.058087074, 'total_time': 0.21700498}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1d8ff-edd7-77eb-8573-47d9350a518c-0', tool_calls=[{'name': 'send_email_tool', 'arg